In [8]:
from google.colab import files
uploaded = files.upload()

Saving Twitter Customer Services.xlsx to Twitter Customer Services.xlsx


In [ ]:
# Cell 1: Environment Setup
!pip install -q transformers nltk spacy scikit-learn seaborn matplotlib pandas plotly

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
nltk.download('vader_lexicon')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

print("Libraries successfully loaded.")

In [ ]:
# Cell 2: Load Real Customer Service Data
import pandas as pd

# 1. Read the Excel file
df = pd.read_excel('Twitter Customer Services.xlsx')

# 2. Filter for incoming customer tweets (inbound = True)
df = df[df['inbound'] == True].copy()

# 3. Map the tweet text column to customer_text
df['customer_text'] = df['text']

# Display first few rows
print(f"Total Customer Tweets Loaded: {len(df)}")
df[['tweet_id', 'author_id', 'created_at', 'customer_text']].head()

In [ ]:
# Cell 3: Data Cleaning
def clean_text(text):
    text = text.lower() # Convert to lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text) # Remove punctuation and numbers
    text = re.sub(r'\s+', ' ', text).strip() # Remove extra spaces
    return text

df['cleaned_text'] = df['customer_text'].apply(clean_text)
df[['customer_text', 'cleaned_text']].head()

In [ ]:
# Cell 4: Sentiment Scoring
sid = SentimentIntensityAnalyzer()

df['compound_score'] = df['cleaned_text'].apply(lambda x: sid.polarity_scores(x)['compound'])

# Categorize sentiment into operational classes
def categorize_sentiment(score):
    if score >= 0.05:
        return 'Positive'
    elif score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

df['predicted_sentiment'] = df['compound_score'].apply(categorize_sentiment)
df[['cleaned_text', 'compound_score', 'predicted_sentiment']]

In [ ]:
# Cell 5: Topic Extraction
tfidf = TfidfVectorizer(max_features=500, stop_words='english')
tfidf_mat = tfidf.fit_transform(df['cleaned_text'])

nmf_model = NMF(n_components=2, random_state=42)
nmf_model.fit(tfidf_mat)

feature_names = tfidf.get_feature_names_out()

for topic_idx, topic in enumerate(nmf_model.components_):
    top_words = [feature_names[i] for i in topic.argsort()[:-5:-1]]
    print(f"Topic {topic_idx + 1} Keywords: {', '.join(top_words)}")

In [ ]:
# Cell 6: Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Assign 'Twitter' as the channel for all tweets
df['channel'] = 'Twitter'

plt.figure(figsize=(8, 4))
sns.countplot(data=df, x='channel', hue='predicted_sentiment', palette='coolwarm')
plt.title('Voice of Customer: Sentiment Distribution Across Support Channels')
plt.xlabel('Support Channel')
plt.ylabel('Ticket Count')
plt.legend(title='Sentiment')
plt.tight_layout()
plt.show()